# 16 — Retry + Circuit Breaker (Amazon FAR style)

Resilience patterns for flaky dependencies. Parts 1-3 build the core (concurrency at Part 3); Parts
4-5 are stretch tasks (breaker-guarded calls, then per-endpoint registries + parallel replay). Clocks
and sleeps are **injected** so tests are deterministic — no real waiting. Fill stubs, run each test
cell, peek at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Retry with exponential backoff

`retry(fn, attempts, base_delay, sleep) -> result`: call `fn()`; on exception, wait
`base_delay * 2**i` before the next try and retry, up to `attempts` total tries; if all fail, re-raise
the last exception. `sleep(seconds)` is an **injected** callable (so tests pass a recorder instead of
really sleeping).

**Lock down:** no sleep after the final attempt; only retry idempotent operations; real systems add
jitter (Part 4 of the breaker family).

In [ ]:
def retry(fn, attempts, base_delay, sleep):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    delays = []
    calls = [0]

    def fn():
        calls[0] += 1
        if calls[0] < 3:
            raise ValueError("transient")
        return "ok"

    assert retry(fn, attempts=5, base_delay=1, sleep=delays.append) == "ok"
    assert calls[0] == 3 and delays == [1, 2]          # slept before the 2nd and 3rd tries
    d2 = []
    try:
        retry(lambda: (_ for _ in ()).throw(RuntimeError()), attempts=3, base_delay=2, sleep=d2.append)
    except RuntimeError:
        pass
    else:
        raise AssertionError("expected the last exception to propagate")
    assert d2 == [2, 4]                                # 2 backoffs between 3 attempts
    print("Part 1 OK")

_t1()

## Part 2 — Circuit breaker state machine

`CircuitBreaker(failure_threshold, reset_timeout)` with `allow(now) -> bool`, `record_success()`,
`record_failure(now)`. States:
- **CLOSED**: calls allowed; `failure_threshold` consecutive failures → **OPEN** (remember `opened_at`).
- **OPEN**: calls rejected until `now - opened_at >= reset_timeout`, then → **HALF_OPEN** (allow a trial).
- **HALF_OPEN**: a trial is allowed; a success → CLOSED (reset), a failure → OPEN again.

**Lock down:** inject `now` (don't read the clock inside); the half-open trial is what lets the breaker
auto-recover.

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold, reset_timeout):
        raise NotImplementedError

    def allow(self, now):
        raise NotImplementedError

    def record_success(self):
        raise NotImplementedError

    def record_failure(self, now):
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    cb = CircuitBreaker(failure_threshold=3, reset_timeout=10)
    assert cb.allow(0)
    cb.record_failure(0); cb.record_failure(1)
    assert cb.allow(2)                                 # 2 failures, still closed
    cb.record_failure(2)                               # 3rd -> OPEN (at t=2)
    assert not cb.allow(3) and not cb.allow(11)        # within reset window
    assert cb.allow(12)                                # 10s elapsed -> HALF_OPEN trial
    cb.record_failure(12)                              # trial failed -> OPEN again
    assert not cb.allow(13)
    assert cb.allow(23)                                # half-open again
    cb.record_success()                                # -> CLOSED
    assert cb.allow(24) and cb.allow(25)
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe breaker

`ThreadSafeBreaker(failure_threshold, reset_timeout)`: same semantics as Part 2 but safe under
concurrent callers (guard the state transitions with a lock), exposing `failures` for inspection.

**The invariant:** with a high threshold (so it stays CLOSED), 8 threads each recording 1000 failures
must leave `failures == 8000` — no lost read-modify-write updates. **Discuss:** the failure counter is
shared mutable state; coarse lock vs atomics; that `allow` itself mutates state (OPEN→HALF_OPEN) so it
needs the lock too.

In [ ]:
import threading


class ThreadSafeBreaker:
    def __init__(self, failure_threshold, reset_timeout):
        raise NotImplementedError

    def allow(self, now):
        raise NotImplementedError

    def record_success(self):
        raise NotImplementedError

    def record_failure(self, now):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    cb = ThreadSafeBreaker(failure_threshold=10_000, reset_timeout=10)

    def worker():
        for _ in range(1000):
            cb.record_failure(0)

    ts = [threading.Thread(target=worker) for _ in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert cb.failures == 8000                         # no lost increments
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Breaker-guarded calls

`guarded_call(breaker, fn, now) -> result`: if `breaker.allow(now)` is False, raise `CircuitOpenError`
**without invoking `fn`** (fail fast, shed load); otherwise run `fn()`, recording success or failure on
the breaker and re-raising on failure.

Also define the `CircuitOpenError` exception.

**Discuss:** fail-fast to protect a struggling dependency, fallbacks / cached responses when open, and
pairing this with Part 1's retry+backoff (retry the *fallback*, not the open breaker).

In [ ]:
class CircuitOpenError(Exception):
    pass


def guarded_call(breaker, fn, now):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    cb = CircuitBreaker(failure_threshold=2, reset_timeout=10)

    def good():
        return "ok"

    def bad():
        raise ValueError("x")

    assert guarded_call(cb, good, 0) == "ok"
    for t in (1, 2):
        try:
            guarded_call(cb, bad, t)
        except ValueError:
            pass
    called = [0]

    def track():
        called[0] += 1
        return "x"

    try:
        guarded_call(cb, track, 3)                      # breaker OPEN -> reject
    except CircuitOpenError:
        pass
    else:
        raise AssertionError("expected CircuitOpenError")
    assert called[0] == 0                               # fn never invoked while open
    assert guarded_call(cb, good, 13) == "ok"           # half-open trial succeeds -> closed
    assert guarded_call(cb, good, 14) == "ok"
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Per-endpoint registry + parallel replay

**(a)** `BreakerRegistry(failure_threshold, reset_timeout)`: a thread-safe **breaker per endpoint**,
created lazily, with `guarded_call(endpoint, fn, now)`. One endpoint tripping must not affect others.

**(b)** `parallel_replay(logs, failure_threshold, reset_timeout, num_procs) -> dict[endpoint, rejected]`:
replay each endpoint's call log through a fresh breaker in parallel with `ProcessPoolExecutor`,
returning how many calls each endpoint would reject. `logs` is `dict[endpoint, list[(now, ok)]]`;
worker is `breaker_workers.replay_calls`.

**Discuss:** per-endpoint isolation (bulkheads), and replaying production logs offline to tune
`failure_threshold` / `reset_timeout`.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import breaker_workers


class BreakerRegistry:
    def __init__(self, failure_threshold, reset_timeout):
        raise NotImplementedError

    def guarded_call(self, endpoint, fn, now):
        raise NotImplementedError


def parallel_replay(logs, failure_threshold, reset_timeout, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    reg = BreakerRegistry(failure_threshold=2, reset_timeout=10)

    def good():
        return "ok"

    def bad():
        raise ValueError()

    assert reg.guarded_call("A", good, 0) == "ok"
    for t in (1, 2):
        try:
            reg.guarded_call("A", bad, t)
        except ValueError:
            pass
    try:
        reg.guarded_call("A", good, 3)                  # A is open
    except CircuitOpenError:
        pass
    else:
        raise AssertionError("expected CircuitOpenError")
    assert reg.guarded_call("B", good, 3) == "ok"       # B is independent
    logs = {
        "x": [(0, False), (1, False), (2, True)],       # opens at t=1, the t=2 call is rejected
        "y": [(0, True), (1, True)],                    # never trips
    }
    assert parallel_replay(logs, 2, 10, 2) == {"x": 1, "y": 0}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Jittered backoff (full/decorrelated jitter) to avoid retry storms; retry budgets.
- Rolling-window failure rate (vs consecutive count); half-open concurrency limit (single probe).
- Bulkheads + timeouts + breaker together; surfacing state to metrics/alerts.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import breaker_workers


def ref_retry(fn, attempts, base_delay, sleep):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:                          # noqa: BLE001
            last = e
            if i < attempts - 1:
                sleep(base_delay * (2 ** i))
    raise last


class RefCircuitBreaker:
    def __init__(self, failure_threshold, reset_timeout):
        self.ft = failure_threshold
        self.rt = reset_timeout
        self.state = "CLOSED"
        self.failures = 0
        self.opened_at = None

    def allow(self, now):
        if self.state == "OPEN":
            if now - self.opened_at >= self.rt:
                self.state = "HALF_OPEN"                 # let one trial through
                return True
            return False
        return True

    def record_success(self):
        self.state = "CLOSED"
        self.failures = 0
        self.opened_at = None

    def record_failure(self, now):
        if self.state == "HALF_OPEN":
            self.state = "OPEN"
            self.opened_at = now
            return
        self.failures += 1
        if self.failures >= self.ft:
            self.state = "OPEN"
            self.opened_at = now


class RefThreadSafeBreaker(RefCircuitBreaker):
    def __init__(self, failure_threshold, reset_timeout):
        super().__init__(failure_threshold, reset_timeout)
        self._lock = threading.Lock()

    def allow(self, now):
        with self._lock:
            return super().allow(now)

    def record_success(self):
        with self._lock:
            super().record_success()

    def record_failure(self, now):
        with self._lock:
            super().record_failure(now)


class CircuitOpenError(Exception):
    pass


def ref_guarded_call(breaker, fn, now):
    if not breaker.allow(now):
        raise CircuitOpenError()
    try:
        r = fn()
    except Exception:                                   # noqa: BLE001
        breaker.record_failure(now)
        raise
    breaker.record_success()
    return r


class RefBreakerRegistry:
    def __init__(self, failure_threshold, reset_timeout):
        self.ft, self.rt = failure_threshold, reset_timeout
        self._b = {}
        self._lock = threading.Lock()

    def _get(self, endpoint):
        with self._lock:
            if endpoint not in self._b:
                self._b[endpoint] = RefThreadSafeBreaker(self.ft, self.rt)
            return self._b[endpoint]

    def guarded_call(self, endpoint, fn, now):
        return ref_guarded_call(self._get(endpoint), fn, now)


def ref_parallel_replay(logs, failure_threshold, reset_timeout, num_procs):
    items = [(ep, log, failure_threshold, reset_timeout) for ep, log in logs.items()]
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return dict(ex.map(breaker_workers.replay_calls, items))


_d = []
assert ref_retry(lambda: 7, attempts=3, base_delay=1, sleep=_d.append) == 7 and _d == []
_cb = RefCircuitBreaker(1, 5); _cb.record_failure(0); assert not _cb.allow(1) and _cb.allow(5)
_cb2 = RefCircuitBreaker(1, 5); _cb2.record_failure(0)
try:
    ref_guarded_call(_cb2, lambda: "x", 1)
except CircuitOpenError:
    pass
else:
    raise AssertionError("expected open")
assert ref_parallel_replay({"e": [(0, False), (0, False), (0, True)]}, 2, 10, 2) == {"e": 1}
print("reference solutions OK")